# Dunnhumby - price elasticity and promotion effectiveness

### Read this top to bottom and you have the whole project

Two years of a grocery retailer's transactions: **2.6M lines, 2,500 households,
92,353 products, 582 stores**, plus **36.8M product-store-week promotion records**
and a coupon campaign log recording which households were targeted.

Three questions:

1. How much does demand move when price moves?
2. What does putting an item on display, or in the mailer, actually do?
3. Did the coupon campaigns make households spend more?

**The third one is where the project earns its keep.** The first answer it gave
was that coupons made people spend less, and most of what follows is about why
that happened and what it took to get a believable number instead.


### The route through

| Part | |
|---|---|
| 1 | The price decomposition everything rests on |
| 2 | Checks: the layer contract, and what the source gets wrong |
| 3 | Is there enough price variation to measure anything? |
| 4 | Elasticity and promotion lift, estimated together |
| 5 | What the elasticities say, and where I do not believe them |
| 6 | The campaign question, answered badly |
| 7 | Parallel trends passes. The answer is still wrong |
| 8 | Looking at the levels |
| 9 | Matching, and what it costs |
| 10 | The answer, and the bias still in it |


In [1]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 44)
# Previews show at least 10 rows throughout. Three is not enough to see a
# pattern, a null cluster, or a grain finer than you assumed.
pd.set_option("display.max_rows", 40)
pd.set_option("display.min_rows", 20)

db = duckdb.connect(str(ROOT / "artifacts" / "dunnhumby.duckdb"), read_only=True)


def q(sql):
    return db.execute(sql).fetchdf()


def show_py(filename, func, lines=None):
    """Print the production code that does something, read at run time.

    Not a copy - the file is read as this cell runs, so what you see is what
    executed.
    """
    path = ROOT / "src" / filename
    rows = path.read_text(encoding="utf-8").splitlines()
    start = next((i for i, r in enumerate(rows)
                  if r.startswith((f"def {func}(", f"{func} = "))), None)
    if start is None:
        print(f"{func} not found in {filename}")
        return
    end = start + 1
    while end < len(rows) and not rows[end].startswith(("def ", "class ", "@")):
        end += 1
    body = rows[start:end]
    if lines:
        body = body[:lines]
    print(f"# {filename}, line {start + 1}\n")
    print("\n".join(body).rstrip())


print("schemas:", ", ".join(
    r[0] for r in db.execute(
        "SELECT DISTINCT schema_name FROM duckdb_tables() ORDER BY 1").fetchall()))


schemas: core, mart, qa


---

## Part 1 - The price decomposition everything rests on

**The problem.** To measure a response to price you need a price, and the source
does not contain one. It contains what the customer paid and three separate
discount columns, all stored negative, with no statement of how they combine.

| | |
|---|---|
| **Built by** | `src/build_warehouse.py` |
| **Written to** | `stg.transactions`, `core.fact_sale` |


| Column | What it is | Who funds it |
|---|---|---|
| `SALES_VALUE` | what the customer paid | - |
| `RETAIL_DISC` | loyalty-card discount | the retailer |
| `COUPON_MATCH_DISC` | the retailer matching a manufacturer coupon | the retailer |
| `COUPON_DISC` | a manufacturer coupon | **the manufacturer** |

The shelf price is what a shopper with no card and no coupon would have paid, so
the two **retailer-funded** discounts are added back:

```sql
base_value = SALES_VALUE - RETAIL_DISC - COUPON_MATCH_DISC
```

`COUPON_DISC` is deliberately **not** added back. The manufacturer funds it, so the
retailer still receives that money - adding it to the shelf price would overstate
what the shopper was asked to pay.

That is a judgement call, not a fact in the data. It is stated here, in the module
docstring and in `docs/assumptions.md`, because every elasticity in this project
inherits it.


In [2]:
q("""
    SELECT product_id, week_number, units,
           ROUND(paid_value, 2)       AS paid,
           ROUND(retailer_discount, 2) AS retail_disc,
           ROUND(manufacturer_coupon, 2) AS mfr_coupon,
           ROUND(retailer_coupon_match, 2) AS coupon_match,
           ROUND(base_value, 2)       AS shelf_value,
           ROUND(unit_paid_price, 3)  AS unit_paid,
           ROUND(unit_base_price, 3)  AS unit_shelf,
           ROUND(discount_depth, 3)   AS depth
    FROM core.fact_sale
    WHERE retailer_discount < 0 AND units BETWEEN 1 AND 4
    ORDER BY paid_value DESC LIMIT 12
""")

,product_id,week_number,units,paid,retail_disc,mfr_coupon,coupon_match,shelf_value,unit_paid,unit_shelf,depth
0,13212959,51,1,499.99,-50.0,0.0,0.0,549.99,499.99,549.99,0.091
1,13213237,66,1,224.99,-75.0,0.0,0.0,299.99,224.99,299.99,0.250
2,9655051,13,1,224.99,-75.0,0.0,0.0,299.99,224.99,299.99,0.250
3,1075007,76,1,209.18,-52.3,0.0,0.0,261.48,209.18,261.48,0.200
4,17242362,87,1,199.99,-100.0,0.0,0.0,299.99,199.99,299.99,0.333
5,13381195,65,1,184.99,-15.0,0.0,0.0,199.99,184.99,199.99,0.075
6,12582517,49,1,149.99,-30.0,0.0,0.0,179.99,149.99,179.99,0.167
7,13381195,67,1,149.99,-50.0,0.0,0.0,199.99,149.99,199.99,0.250
8,12582517,68,1,149.99,-30.0,0.0,0.0,179.99,149.99,179.99,0.167
9,12582517,43,1,149.99,-30.0,0.0,0.0,179.99,149.99,179.99,0.167


> **Conclusion.** Nothing downstream works until this is decided, and the source decides none of it. Writing the rule down - including the part that is a choice rather than a derivation - is the difference between an elasticity someone can check and one they have to trust.


---

## Part 2 - Checks: the layer contract, and what the source gets wrong

**The contract.** `stg` may not lose a row. `core` may not change the row count of
the fact it is built from. `mart` may not introduce a number that cannot be
derived from `core`.

Stated before the SQL was written, then tested.

| | |
|---|---|
| **Built by** | `src/build_warehouse.py` - `build_qa()` |
| **Written to** | `qa.layer_contract`, `qa.source_quality` |


In [3]:
q("SELECT * FROM qa.layer_contract")

,step,rows_in,rows_out,delta
0,raw -> stg: transactions,2595732,2595732,0
1,stg -> core: fact_sale,2595732,2595732,0
2,core -> mart: revenue ties,7447281,7447281,0


Zero at every step, and the revenue ties to the dollar between `core` and `mart`.

### What the source gets wrong


In [4]:
q("SELECT * FROM qa.source_quality ORDER BY rows_affected DESC")

,issue,rows_affected,rows_total,treatment
0,units above 20 (weighted or bulk),23445,2595732,Produce sold by weight. Units are not co...
1,paid value not positive,18850,2595732,Returns and adjustments. Excluded from p...
2,units is zero,14466,2595732,A line with no units has no unit price. ...
3,retailer discount stored POSITIVE,36,2595732,A discount that raises the price. Anomal...
4,base value below paid value,35,2595732,Shelf price below what was paid. Impossi...


Four of these are ordinary and handled by exclusion. **One is an anomaly worth
naming:** 36 rows store the retailer discount as a *positive* number - a discount
that raises the price.

It is 36 rows in 2.6 million, so it changes nothing. The reason to report it is
that a reader who finds it themselves will wonder what else went unexamined.


In [5]:
q("""
    SELECT COUNT(*) FILTER (WHERE usable_for_pricing)        AS usable_lines,
           COUNT(*)                                          AS total_lines,
           ROUND(100.0 * COUNT(*) FILTER (WHERE usable_for_pricing)
               / COUNT(*), 1)                                AS pct_of_lines,
           ROUND(100.0 * SUM(paid_value) FILTER (WHERE usable_for_pricing)
               / SUM(paid_value), 1)                         AS pct_of_revenue
    FROM core.fact_sale
""")

,usable_lines,total_lines,pct_of_lines,pct_of_revenue
0,2553365,2595732,98.4,92.4


> **Conclusion.** 98.4% of lines and 92.4% of revenue survive into the pricing analysis. The exclusions are flagged on the row rather than filtered away in a WHERE clause, so they stay countable.


---

## Part 3 - Is there enough price variation to measure anything?

**The question before the question.** Elasticity is identified by a product being
cheaper than its *own* normal price. If prices barely move, there is nothing to
estimate from, and a regression will return a number anyway.

This is the check that decides whether the rest of the project is possible - and
it is the check that TheLook failed, which is why the elasticity work is here and
not there.

| | |
|---|---|
| **Built by** | `mart.product_week` |
| **Written to** | - |


In [6]:
q("""
    WITH v AS (
        SELECT product_id, commodity, COUNT(*) AS weeks,
               STDDEV(LN(unit_paid_price)) AS sd_log_price,
               AVG(discount_depth)         AS avg_depth
        FROM mart.product_week
        WHERE unit_paid_price > 0 AND units > 0
        GROUP BY 1, 2 HAVING COUNT(*) >= 20
    )
    SELECT COUNT(*)                                          AS products,
           ROUND(AVG(sd_log_price), 4)                       AS mean_sd_log_price,
           COUNT(*) FILTER (WHERE sd_log_price > 0.05)       AS with_real_variation,
           ROUND(100.0 * COUNT(*) FILTER (WHERE sd_log_price > 0.05)
               / COUNT(*), 1)                                AS pct,
           ROUND(AVG(avg_depth), 4)                          AS mean_discount_depth
    FROM v
""")

,products,mean_sd_log_price,with_real_variation,pct,mean_discount_depth
0,16310,0.1425,14170,86.9,0.1114


### Which categories actually discount


In [7]:
q("""
    SELECT commodity,
           COUNT(DISTINCT product_id)        AS products,
           COUNT(*)                          AS product_weeks,
           ROUND(SUM(revenue))               AS revenue,
           ROUND(AVG(discount_depth), 3)     AS avg_discount_depth
    FROM mart.product_week WHERE commodity IS NOT NULL
    GROUP BY 1 ORDER BY revenue DESC LIMIT 14
""")

,commodity,products,product_weeks,revenue,avg_discount_depth
0,SOFT DRINKS,1696,32692,326535.0,0.132
1,BEEF,1108,11961,311904.0,0.063
2,FLUID MILK PRODUCTS,455,8814,205303.0,0.094
3,CHEESE,1096,22750,189353.0,0.173
4,FRZN MEAT/MEAT DINNERS,1268,28882,160496.0,0.185
5,BAG SNACKS,1521,24070,148320.0,0.086
6,BEERS/ALES,793,10557,147344.0,0.001
7,FROZEN PIZZA,900,19189,145924.0,0.135
8,BAKED BREAD/BUNS/ROLLS,1169,20098,145908.0,0.147
9,COLD CEREAL,638,14855,111702.0,0.107


> **Conclusion.** 14,170 of 16,310 products carry real price variation, and mean discount depth is 11%. Note BEERS/ALES and CIGARETTES at ~0% depth - price-controlled categories where no elasticity can be identified, whatever a regression returns.


---

## Part 4 - Elasticity and promotion lift, estimated together

**The model.** On a product x week panel, per commodity:

```
log(units) = b*log(price) + g*display + d*mailer + product FE + week FE + e
```

All three regressors in one equation, deliberately.

| | |
|---|---|
| **Built by** | `src/elasticity.py` |
| **Written to** | `mart.price_elasticity` - one row per commodity |


### Why together

Price, display and mailer move at the same time - a retailer discounts in the same
week it features. Estimate price alone and it absorbs the display effect, and the
elasticity comes out too large.

### Why fixed effects

| | |
|---|---|
| **product** | absorbs the level difference between a 40-cent item and a 12-dollar one, so the estimate comes from a product being cheaper than its *own* normal price |
| **week** | absorbs anything that moved every product at once - a holiday, a pay week, the weather |

Both are absorbed by alternating-projection demeaning rather than dummies, because
a commodity can contain two thousand products across a hundred weeks.


In [8]:
show_py("elasticity.py", "absorb")

# elasticity.py, line 75

def absorb(frame: pd.DataFrame, columns: list[str], keys: list[str]) -> pd.DataFrame:
    """Remove the fixed effects in `keys` from `columns` by alternating projection."""
    out = frame[columns].astype(float).copy()
    for _ in range(DEMEAN_PASSES):
        before = out.to_numpy(copy=True)
        for key in keys:
            out = out - out.groupby(frame[key].to_numpy()).transform("mean")
        if np.nanmax(np.abs(out.to_numpy() - before)) < 1e-10:
            break
    return out


In [9]:
show_py("elasticity.py", "fit", lines=34)

# elasticity.py, line 87

def fit(panel: pd.DataFrame) -> dict[str, float] | None:
    """OLS of demeaned log-units on demeaned regressors, with robust errors."""
    regressors = ["log_price", "display_intensity", "mailer_intensity"]
    data = panel.dropna(subset=["log_units", *regressors])
    if len(data) < MIN_ROWS_PER_COMMODITY:
        return None

    demeaned = absorb(data, ["log_units", *regressors], ["product_id", "week_number"])
    y = demeaned["log_units"].to_numpy()
    X = demeaned[regressors].to_numpy()
    if np.linalg.matrix_rank(X) < X.shape[1] or np.allclose(X[:, 0].std(), 0):
        return None

    xtx_inv = np.linalg.pinv(X.T @ X)
    beta = xtx_inv @ (X.T @ y)
    residual = y - X @ beta

    # Degrees of freedom: the absorbed effects cost one parameter each.
    absorbed = data["product_id"].nunique() + data["week_number"].nunique() - 1
    dof = max(len(data) - X.shape[1] - absorbed, 1)

    # HC1 heteroskedasticity-robust covariance. Prices and volumes are 

> **Conclusion.** The standard errors are heteroskedasticity-robust and the degrees of freedom are charged for every absorbed effect. Prices and volumes are both heavily skewed, so classical errors here would be too narrow and every t-statistic too confident.


---

## Part 5 - What the elasticities say, and where I do not believe them

147 commodities estimated, 134 of them statistically significant.

| | |
|---|---|
| **Built by** | `src/elasticity.py` |
| **Written to** | `mart.price_elasticity` |


In [10]:
q("""
    SELECT COUNT(*)                                        AS commodities,
           COUNT(*) FILTER (WHERE elasticity < -1)         AS elastic,
           COUNT(*) FILTER (WHERE elasticity BETWEEN -1 AND 0) AS inelastic,
           COUNT(*) FILTER (WHERE elasticity > 0)          AS wrong_sign,
           COUNT(*) FILTER (WHERE ABS(elasticity_t) > 2)   AS significant,
           ROUND(quantile_cont(elasticity, 0.25), 2)       AS p25,
           ROUND(quantile_cont(elasticity, 0.50), 2)       AS median,
           ROUND(quantile_cont(elasticity, 0.75), 2)       AS p75,
           ROUND(AVG(display_lift_pct), 1)                 AS mean_display_lift,
           ROUND(AVG(mailer_lift_pct), 1)                  AS mean_mailer_lift
    FROM mart.price_elasticity
""")

,commodities,elastic,inelastic,wrong_sign,significant,p25,median,p75,mean_display_lift,mean_mailer_lift
0,147,33,112,2,134,-0.94,-0.71,-0.41,7.4,42.9


### The most and least price-sensitive categories


In [11]:
q("""
    SELECT commodity,
           ROUND(elasticity, 2)        AS elasticity,
           ROUND(elasticity_t, 1)      AS t_stat,
           ROUND(display_lift_pct, 1)  AS display_lift_pct,
           ROUND(mailer_lift_pct, 1)   AS mailer_lift_pct,
           ROUND(revenue)              AS revenue,
           observations
    FROM mart.price_elasticity
    WHERE revenue > 25000 AND NOT implausible_sign
    ORDER BY elasticity LIMIT 14
""")

,commodity,elasticity,t_stat,display_lift_pct,mailer_lift_pct,revenue,observations
0,BACON,-1.77,-18.7,8.9,298.7,43803.0,2741
1,REFRGRATD JUICES/DRNKS,-1.71,-32.0,4.0,26.1,66997.0,7179
2,PAPER TOWELS,-1.63,-10.5,12.6,96.2,28980.0,2438
3,HOT DOGS,-1.62,-21.8,16.7,79.7,31841.0,2684
4,COLD CEREAL,-1.51,-25.2,11.5,45.6,107766.0,13914
5,SALAD MIX,-1.48,-11.9,15.1,56.9,43241.0,3270
6,BATH TISSUES,-1.44,-9.2,15.6,87.1,57703.0,3167
7,LUNCHMEAT,-1.39,-31.7,7.7,62.7,84739.0,12891
8,MEAT - MISC,-1.37,-13.3,15.1,62.5,62289.0,2611
9,CANDY - CHECKLANE,-1.34,-36.6,3.6,61.9,25971.0,8974


### The pattern is the commercial point

**Stockpilable, branded, frequently promoted goods are elastic.** Bacon at −1.77,
refrigerated juice at −1.71, cold cereal at −1.51. A shopper can buy bacon this
week instead of next, so a price cut mostly moves *when* they buy.

**Fresh meat is inelastic on price but enormous on the mailer.** Beef: elasticity
−0.24, mailer lift **+161%**. It is a traffic driver - bought because the trip
happened, not because the price fell.

Those two facts imply opposite promotional strategies, and the model can separate
them only because display and mailer sit in the equation beside price.


### Where I do not believe the estimate


In [12]:
q("""
    SELECT commodity, ROUND(elasticity, 3) AS elasticity,
           ROUND(elasticity_t, 2) AS t_stat, ROUND(revenue) AS revenue,
           observations
    FROM mart.price_elasticity
    WHERE revenue > 20000 AND elasticity > -0.12
    ORDER BY elasticity DESC LIMIT 12
""")

,commodity,elasticity,t_stat,revenue,observations
0,DELI MEATS,0.006,0.25,100449.0,6552
1,POTATOES,-0.002,-0.02,43988.0,1612
2,VEGETABLES - ALL OTHERS,-0.006,-0.18,32006.0,2999
3,APPLES,-0.013,-0.12,36541.0,2037
4,CHEESES,-0.055,-1.90,41371.0,4454
5,SALADS/DIPS,-0.086,-3.22,21692.0,3792
6,CHICKEN/POULTRY,-0.087,-2.05,29964.0,1906


> **Conclusion.** Potatoes, apples and other vegetables return elasticities indistinguishable from zero. That is almost certainly **measurement, not behaviour**: produce is sold by weight, so 'units' and therefore unit price are not comparable between baskets. The 20-unit filter removes the worst of it and does not fix it. Reported with the flag rather than dropped.


### The assumption underneath all of it

Within a product and a week, the remaining price variation is assumed unrelated to
demand. **It is not, and the direction is knowable:** retailers discount what they
expect to sell, so a demand spike the retailer anticipated gets attributed to the
price cut.

Every elasticity here should be read as an **upper bound on sensitivity**. The
display and mailer controls remove the main channel through which that happens -
promotion - but not all of it.


---

## Part 6 - The campaign question, answered badly

**The question.** 2,500 households, 1,584 of whom received at least one coupon
campaign. Did the campaigns make them spend more?

916 households received nothing. That is a comparison group that exists in the
data rather than one assembled by assumption - which is rare, and is why this
dataset is worth the trouble.

| | |
|---|---|
| **Built by** | `src/campaign_effect.py` |
| **Written to** | `mart.campaign_effect` |


**The estimator.** Difference-in-differences on weekly household spend:

```
effect = (treated after - treated before) - (control after - control before)
```

Subtracting the control's own change removes anything that moved every household
at once - a holiday, a price war, the season.


In [13]:
q("""
    SELECT campaign_id, campaign_type, treated_households,
           ROUND(effect_per_week, 2)        AS effect_per_week,
           ROUND(effect_se, 2)              AS std_error,
           ROUND(effect_t, 2)               AS t_stat,
           ROUND(effect_pct_of_baseline, 1) AS pct_of_baseline
    FROM mart.campaign_effect ORDER BY effect_t
""")

,campaign_id,campaign_type,treated_households,effect_per_week,std_error,t_stat,pct_of_baseline
0,11,TypeB,214,-6.87,2.15,-3.20,-8.7
1,25,TypeB,187,-8.12,2.62,-3.10,-8.2
2,23,TypeB,183,-7.06,2.71,-2.60,-7.1
3,17,TypeB,202,-5.24,2.13,-2.46,-6.5
4,14,TypeC,224,-5.74,2.36,-2.43,-6.9
5,16,TypeB,188,-5.10,2.41,-2.12,-6.0
6,9,TypeB,176,-3.20,2.04,-1.57,-5.1
7,12,TypeB,170,-3.04,2.28,-1.34,-3.7
8,13,TypeA,1077,-0.86,0.96,-0.89,-1.5
9,30,TypeA,361,-1.11,1.49,-0.75,-2.1


> **Conclusion.** **Twelve of 17 campaigns show a negative effect, six of them significantly so** - up to -8.7% of baseline weekly spend. Coupons do not make people spend less. The estimate is not a surprising finding, it is a broken one, and the rest of this notebook is about finding out how.


---

## Part 7 - Parallel trends passes. The answer is still wrong

**The standard check.** Difference-in-differences is valid only if the two groups
were on parallel trends beforehand - not at the same level, at the same *slope*.

That is testable: regress the weekly treated-minus-control gap on time over the
eight weeks before the campaign and see whether the slope differs from zero.

| | |
|---|---|
| **Built by** | `src/campaign_effect.py` - `parallel_trends()` |
| **Written to** | `mart.campaign_effect` |


In [14]:
show_py("campaign_effect.py", "parallel_trends")

# campaign_effect.py, line 186

def parallel_trends(panel: pd.DataFrame) -> dict[str, float]:
    """Test whether the groups were already diverging before the campaign.

    Regresses the weekly treated-minus-control gap on time over the pre-period.
    A slope indistinguishable from zero supports the assumption; a significant
    slope means the difference-in-differences estimate is not interpretable.
    """
    pre = panel[panel["weeks_from_start"] < 0]
    weekly = (
        pre.groupby(["weeks_from_start", "is_treated"])["spend"].mean().unstack("is_treated")
    )
    weekly.columns = ["control", "treated"]
    weekly["gap"] = weekly["treated"] - weekly["control"]

    x = weekly.index.to_numpy(dtype=float)
    y = weekly["gap"].to_numpy(dtype=float)
    design = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(design, y, rcond=None)
    residual = y - design @ beta
    dof = max(len(x) - 2, 1)
    sigma2 = float((residual ** 2).sum() / dof)
    se = float(np.s

In [15]:
q("""
    SELECT campaign_id, campaign_type,
           ROUND(pre_trend_slope, 3) AS pre_trend_slope,
           ROUND(pre_trend_t, 2)     AS pre_trend_t,
           pre_trend_passes,
           ROUND(effect_per_week, 2) AS naive_effect,
           ROUND(effect_t, 2)        AS naive_t
    FROM mart.campaign_effect ORDER BY campaign_id
""")

,campaign_id,campaign_type,pre_trend_slope,pre_trend_t,pre_trend_passes,naive_effect,naive_t
0,5,TypeB,-0.213,-0.21,True,-1.84,-0.67
1,7,TypeB,-0.211,-0.34,True,1.14,0.57
2,8,TypeA,-0.310,-2.15,False,0.03,0.03
3,9,TypeB,0.568,0.88,True,-3.20,-1.57
4,11,TypeB,-1.052,-0.92,True,-6.87,-3.20
5,12,TypeB,-0.651,-1.14,True,-3.04,-1.34
6,13,TypeA,-0.591,-1.41,True,-0.86,-0.89
7,14,TypeC,-1.204,-0.92,True,-5.74,-2.43
8,16,TypeB,-1.014,-2.03,False,-5.10,-2.12
9,17,TypeB,-0.547,-0.87,True,-5.24,-2.46


> **Conclusion.** **Fifteen of 17 pass.** The standard check gives the green light on exactly the campaigns producing an impossible answer. A test that passes while the result is wrong is more dangerous than no test, because it ends the enquiry.


---

## Part 8 - Looking at the levels

**The diagnostic that caught it was not clever.** Parallel trends is a statement
about slopes. It says nothing about whether the two groups are comparable at all.

So: how much did each group spend before the campaign started?

| | |
|---|---|
| **Built by** | `mart.campaign_effect` |
| **Written to** | - |


In [16]:
q("""
    SELECT campaign_id, campaign_type, treated_households,
           ROUND(treated_before, 2)  AS treated_spend_before,
           ROUND(control_before, 2)  AS control_spend_before,
           ROUND(baseline_ratio, 1)  AS ratio,
           ROUND(effect_per_week, 2) AS naive_effect,
           pre_trend_passes
    FROM mart.campaign_effect ORDER BY baseline_ratio DESC
""")

,campaign_id,campaign_type,treated_households,treated_spend_before,control_spend_before,ratio,naive_effect,pre_trend_passes
0,5,TypeB,166,85.93,8.48,10.1,-1.84,True
1,12,TypeB,170,81.16,9.78,8.3,-3.04,True
2,7,TypeB,198,72.12,8.82,8.2,1.14,True
3,11,TypeB,214,79.38,9.78,8.1,-6.87,True
4,14,TypeC,224,83.09,10.70,7.8,-5.74,True
5,16,TypeB,188,84.45,11.12,7.6,-5.10,False
6,25,TypeB,187,99.65,14.11,7.1,-8.12,True
7,23,TypeB,183,99.95,14.30,7.0,-7.06,True
8,17,TypeB,202,81.08,11.97,6.8,-5.24,True
9,30,TypeA,361,53.77,8.17,6.6,-1.11,True


### Between four and ten times

The treated households spend **$44 to $100 a week**. The comparison group spends
**$8 to $14**. The average ratio is **7.0x**.

Dunnhumby targeted these campaigns at their heaviest shoppers. The 916 households
who received nothing are not a control group - they are a different population.

And a group averaging $10 a week has so little variance that its slope will look
parallel to almost anything. **That is why the test passed.** It was not wrong; it
was answering a question about slopes while the problem was about comparability.

What the naive estimate measured was heavy shoppers reverting toward their own
mean, with the campaign happening to sit at the peak.


> **Conclusion.** A diagnostic that passes is not evidence the design is sound - only that one specific failure did not occur. The habit worth taking from this is to look at the levels of both groups before trusting any difference between them.


---

## Part 9 - Matching, and what it costs

**The fix.** Build a comparison group that starts from the same place: for each
treated household, find a never-treated household with similar pre-period weekly
spend, inside a caliper of 0.25 standard deviations.

Treated households with no control inside the caliper are **dropped**, and the
number dropped is reported - it measures how far outside the comparison group's
range the targeting went.

| | |
|---|---|
| **Built by** | `src/campaign_effect.py` - `match_on_pre_spend()` |
| **Written to** | `mart.campaign_effect` |


In [17]:
show_py("campaign_effect.py", "match_on_pre_spend", lines=42)

# campaign_effect.py, line 142

def match_on_pre_spend(household: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, dict]:
    """Nearest-neighbour match on pre-period spend, inside a caliper.

    The naive comparison fails because the campaigns were targeted: treated
    households already spent far more than the never-treated ones. Matching
    rebuilds a comparison group that starts from the same place.

    Treated households with no control inside the caliper are DROPPED. The
    number dropped is the honest measure of how far outside the comparison
    group's range the targeting went, and it is reported rather than absorbed.
    """
    treated = household.loc[household["is_treated"]].copy()
    control = household.loc[~household["is_treated"]].copy()
    caliper = CALIPER_SD * float(household["before"].std(ddof=1))

    control_sorted = control.sort_values("before").reset_index()
    values = control_sorted["before"].to_numpy()

    use_count = np.zeros(len(control_sorted))
    

### Did it work? The test is whether the two groups now start level.


In [18]:
q("""
    SELECT campaign_id, campaign_type,
           matched_treated_matched  AS treated_kept,
           matched_treated_dropped  AS treated_dropped,
           matched_controls_used    AS controls_used,
           ROUND(matched_treated_before, 2) AS treated_before,
           ROUND(matched_control_before, 2) AS control_before,
           ROUND(matched_treated_before - matched_control_before, 3) AS balance_gap
    FROM mart.campaign_effect ORDER BY campaign_id
""")

,campaign_id,campaign_type,treated_kept,treated_dropped,controls_used,treated_before,control_before,balance_gap
0,5,TypeB,122,44,46,60.39,60.16,0.235
1,7,TypeB,157,41,63,55.68,55.28,0.406
2,8,TypeA,1015,61,230,49.83,49.74,0.091
3,9,TypeB,159,17,72,56.04,55.48,0.562
4,11,TypeB,189,25,71,66.70,66.59,0.110
5,12,TypeB,149,21,62,67.17,66.89,0.280
6,13,TypeA,1025,52,250,51.91,51.74,0.165
7,14,TypeC,180,44,83,61.63,61.64,-0.011
8,16,TypeB,156,32,76,63.66,63.88,-0.225
9,17,TypeB,177,25,80,66.76,66.47,0.292


> **Conclusion.** Balance within **$0.22 on bases of $44-76**, against a 7x gap before. The cost is **612 of 6,407 treated households dropped** for having no comparable control - 9.6%, and the honest measure of how targeted these campaigns were.


---

## Part 10 - The answer, and the bias still in it

Same estimator, same households, same weeks. Only the comparison group changed.

| | |
|---|---|
| **Built by** | `src/campaign_effect.py` |
| **Written to** | `mart.campaign_effect` |


In [19]:
q("""
    SELECT campaign_id, campaign_type,
           ROUND(effect_per_week, 2)          AS naive_effect,
           ROUND(effect_t, 2)                 AS naive_t,
           ROUND(matched_effect_per_week, 2)  AS matched_effect,
           ROUND(matched_effect_t, 2)         AS matched_t,
           ROUND(matched_effect_pct, 1)       AS matched_pct_of_baseline
    FROM mart.campaign_effect ORDER BY matched_effect_pct DESC
""")

,campaign_id,campaign_type,naive_effect,naive_t,matched_effect,matched_t,matched_pct_of_baseline
0,26,TypeA,1.95,1.27,19.51,6.78,41.4
1,30,TypeA,-1.11,-0.75,16.80,4.22,38.4
2,20,TypeC,4.02,1.63,23.26,4.75,36.2
3,7,TypeB,1.14,0.57,18.24,3.31,32.8
4,23,TypeB,-7.06,-2.60,23.50,4.41,30.8
5,22,TypeB,-0.28,-0.16,18.21,4.35,28.0
6,25,TypeB,-8.12,-3.10,19.59,3.01,26.0
7,14,TypeC,-5.74,-2.43,15.62,3.17,25.3
8,9,TypeB,-3.20,-1.57,13.31,2.40,23.7
9,8,TypeA,0.03,0.03,10.94,2.81,22.0


In [20]:
q("""
    SELECT COUNT(*) FILTER (WHERE effect_per_week < 0)        AS naive_negative,
           COUNT(*) FILTER (WHERE effect_t < -2)              AS naive_sig_negative,
           COUNT(*) FILTER (WHERE matched_effect_per_week > 0) AS matched_positive,
           COUNT(*) FILTER (WHERE matched_effect_t > 2)       AS matched_sig_positive,
           ROUND(quantile_cont(matched_effect_pct, 0.5), 1)   AS median_pct_lift
    FROM mart.campaign_effect
""")

,naive_negative,naive_sig_negative,matched_positive,matched_sig_positive,median_pct_lift
0,12,6,17,13,23.7


### Every sign flips

| | Naive | Matched |
|---|---|---|
| Negative effects | **12 of 17** | 0 |
| Significantly negative | **6** | 0 |
| Significantly positive | 0 | **13** |
| Median effect | negative | **+23.7% of baseline** |

### What I would still not claim

**Matching fixes selection on what you can see.** It does nothing about selection
on what you cannot.

And there is a bias running the *other* way that is easy to miss. Matching on a
single noisy pre-period measure selects control households who happened to be
having a good spell at that level - and they revert **down**, which flatters the
treated group.

So the naive estimate is biased low and the matched estimate is probably biased
high. The true effect is somewhere between them, nearer the matched one.

**A randomised holdout would settle it in one campaign**, and that is the
recommendation: hold out 10% of the targeted list, send them nothing, and the next
campaign answers this question properly instead of by argument.


> **Conclusion.** The deliverable is not '+23.7%'. It is that the first answer was negative, the standard check endorsed it, and it took looking at the levels to find out why - and that the corrected answer still carries a bias in the opposite direction which only an experiment removes.


---

## What this project demonstrates

| | |
|---|---|
| **Price economics** | own-price elasticity for 147 categories, with display and mailer lift separated from it rather than absorbed into it |
| **Panel econometrics** | two-way fixed effects absorbed by alternating projection, robust standard errors, degrees of freedom charged for every absorbed effect |
| **Causal inference** | difference-in-differences, a parallel-trends test, and the recognition that passing it was not enough |
| **Diagnosis** | an implausible result traced to its cause rather than reported or discarded |
| **Scope discipline** | elasticities labelled as an upper bound, produce flagged as measurement error, both directions of remaining bias stated |

### Limitations, stated

- Elasticities are an **upper bound on sensitivity**: retailers discount what they
  expect to sell, and the controls remove the main channel but not all of it.
- Fresh produce elasticities are **measurement error**, not behaviour. Sold by weight.
- Two of 147 commodities return a positive sign and are flagged, not reported.
- The matched campaign estimate is **probably biased high**, for the reason in Part 10.
- Promotion records cover weeks 9-101 of 102, and 43% of product-weeks.
- `COUPON_DISC` is excluded from the shelf price. That is a judgement call.

---

*Assumptions: `docs/assumptions.md` - Method: `docs/method.md`*


In [21]:
db.close()
print("Walkthrough complete.")

Walkthrough complete.
